In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Classification

In [ ]:
import pandas as pd
import re

!pip install langdetect

from langdetect import detect

# =========================================================
# 1. read data
# =========================================================
df = pd.read_csv("/content/zong.csv", encoding="latin1")

# =========================================================
# 2.
# =========================================================
for col in ["review", "title", "place_title", "name", "date", "images", "local_guide"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

if "likes" in df.columns:
    df["likes"] = pd.to_numeric(df["likes"], errors="coerce").fillna(0)

if "rating" in df.columns:
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# =========================================================
# 3. Keyword Library
# =========================================================

# ---- Residents (Italian) Keywords ----
resident_keywords_it = [
    "sempre vengo", "vengo spesso", "locale", "del posto", "da anni",
    "da tanto tempo", "abituale", "clienti abituali", "vicinato",
    "quartiere", "prezzo aumentato", "conosco il proprietario",
    "da bambino", "nostro posto", "il solito posto"
]

# ---- Tourists (Italian + English) Keywords ----
tourist_keywords_it_en = [
    # Italian
    "prima volta", "turista", "vacanza", "da visitare", "foto",
    "fare foto", "itinerario", "giro turistico", "luogo famoso",
    # English
    "first time", "trip", "must see", "holiday", "tour", "photo",
    "guide", "check in", "check-in", "instagrammable", "bucket list"
]

# ---- tourist tone ----
tourist_keywords_extra = [
    r"first time", r"must visit", r"bucket list", r"trip", r"travel",
    r"guidebook", r"amazing view", r"stunning", r"tour", r"crowded",
    r"worth visiting", r"postcard", r"beautiful place", r"photo spot",
    r"as a tourist", r"wander", r"explore"
]

# ---- resident tone ----
resident_keywords_extra = [
    r"come here often", r"usually", r"every week", r"every day", r"daily",
    r"my neighborhood", r"near my home", r"local", r"our area",
    r"after work", r"lunch break", r"weekday", r"family dinner",
    r"used to be better", r"price increased", r"regular", r"grocery",
    r"my usual place"
]

# =========================================================
# 4. Time of year: peak tourist season
# =========================================================
tourist_months = [6, 7, 8, 9, 12]

# =========================================================
# 5. Long-term / short-term contextual characteristics
# =========================================================
resident_context_patterns = [
    r"\b(i have been|been going|coming here) for \d+ years",
    r"\balways come\b",
    r"\blocal\b",
    r"\bregular\b",
]

tourist_context_patterns = [
    r"\btravel\b",
    r"\bon vacation\b",
    r"\btrip\b",
    r"\bitinerary\b",
    r"\bvisited\b",
]

# =========================================================
# 6. Location type characteristics
# =========================================================
tourist_place_types = [
    "church", "cathedral", "museum", "palace", "bridge",
    "attraction", "square", "monument", "landmark", "tower",
    "gallery", "tour", "boat", "gondola"
]

resident_place_types = [
    "supermarket", "pharmacy", "market", "bakery", "cafe",
    "restaurant", "bar", "laundry", "shop"
]

# =========================================================
# 7.
# =========================================================
def contains_keywords(text, keyword_list):
    text = str(text).lower()
    return any(re.search(k, text) for k in keyword_list)

#
if "name" in df.columns:
    user_review_counts = df["name"].value_counts().to_dict()
else:
    user_review_counts = {}

# =========================================================
# 8. combine
# =========================================================
def classify_merged(row):
    review = str(row.get("review", "")).lower().strip()
    title = str(row.get("title", "")).lower().strip()
    place = str(row.get("place_title", "")).lower().strip()
    date_str = row.get("date", "")
    full_text = (review + " " + title).strip()

    likes = row.get("likes", 0)
    rating = row.get("rating", None)
    images = str(row.get("images", ""))
    local_guide_raw = str(row.get("local_guide", "")).lower()
    is_local_guide = local_guide_raw in ["1", "true", "yes"]
    user_name = str(row.get("name", ""))

    resident_score = 0
    tourist_score = 0

    # --------------------------
    # A. Language detection
    # If not 'it' or 'en', more likely to be a visitor
    # --------------------------
    try:
        lang = detect(full_text if full_text else review)
        if lang not in ["it", "en"]:
            tourist_score += 2
    except:
        pass

    # --------------------------
    # B.
    # --------------------------
    for kw in resident_keywords_it:
        if kw in full_text:
            resident_score += 2

    for kw in tourist_keywords_it_en:
        if kw in full_text:
            tourist_score += 2

    # --------------------------
    # C.
    # --------------------------
    if contains_keywords(full_text, resident_keywords_extra):
        resident_score += 2

    if contains_keywords(full_text, tourist_keywords_extra):
        tourist_score += 2

    # --------------------------
    # D. long time/short time
    # --------------------------
    for pat in resident_context_patterns:
        if re.search(pat, full_text):
            resident_score += 2

    for pat in tourist_context_patterns:
        if re.search(pat, full_text):
            tourist_score += 2

    # --------------------------
    # E. Comment length\
    # Longer comments tend to be from residents
    # --------------------------
    if len(full_text.split()) > 80:
        resident_score += 1

    # --------------------------
    # F. Time of year: peak tourist season
    # --------------------------
    try:
        month = pd.to_datetime(date_str).month
        if month in tourist_months:
            tourist_score += 1
    except:
        pass

    # --------------------------
    # G. Location type
    # --------------------------
    if any(w in place for w in tourist_place_types):
        tourist_score += 2

    if any(w in place for w in resident_place_types):
        resident_score += 1

    # --------------------------
    # H. Local Guide behavior
    # --------------------------
    if is_local_guide and likes > 10:
        tourist_score += 1

    # --------------------------
    # I. High ratings + lots of photos
    # --------------------------
    if pd.notna(rating) and rating == 5:
        tourist_score += 1

    if images.count("http") >= 3:
        tourist_score += 1

    # --------------------------
    # J. Users only appear once; they are more like visitors
    # --------------------------
    if user_name and user_review_counts.get(user_name, 0) == 1:
        tourist_score += 1

    # --------------------------
    # K. Final
    # --------------------------
    if resident_score > tourist_score:
        return "Resident"
    else:
        return "Tourist"

# =========================================================
# 9. Classify
# =========================================================
df["label"] = df.apply(classify_merged, axis=1)

# =========================================================
# 10. Output
# =========================================================
output_file = "/content/classified_output_merged.csv"
df.to_csv(output_file, index=False)

print("classification done")
print(f"output：{output_file}")
print(df[["review", "label"]].head(20))

**CSV to GEOJSON**

In [ ]:
import pandas as pd
import json
import re
from math import isfinite

float_re = re.compile(r'[-+]?\d*\.\d+|[-+]?\d+')

def find_two_floats(s):
    if pd.isna(s):
        return None
    s = str(s)
    # match two floats in string
    nums = float_re.findall(s)
    if len(nums) >= 2:
        try:
            a = float(nums[0])
            b = float(nums[1])
            return (a, b)
        except:
            return None
    return None

def parse_wkt_point(s):
    # accepts "POINT(lon lat)" or "POINT(lat lon)"
    if not isinstance(s, str):
        return None
    m = re.search(r'POINT\s*\(\s*([^\s,]+)[\s,]+([^\s,]+)\s*\)', s, re.IGNORECASE)
    if m:
        try:
            a = float(m.group(1))
            b = float(m.group(2))
            return (a, b)
        except:
            return None
    return None

def valid_lat(x):
    return isfinite(x) and -90.0 <= x <= 90.0

def valid_lon(x):
    return isfinite(x) and -180.0 <= x <= 180.0

def detect_and_get_lon_lat(row, gps_col, lat_col, lon_col, lon_first_flag):
    # priority: explicit lat/lon columns > WKT > gps string
    if lat_col and lon_col and lat_col in row.index and lon_col in row.index:
        lat = row[lat_col]
        lon = row[lon_col]
        try:
            lat = float(lat)
            lon = float(lon)
            return lon, lat
        except:
            return None

    # try WKT
    if gps_col and gps_col in row.index:
        s = row[gps_col]
        p = parse_wkt_point(s)
        if p:
            a, b = p
            if lon_first_flag:
                return a, b
            # heuristic: if first is lon-range and second lat-range, treat as lon,lat
            if valid_lon(a) and valid_lat(b):
                return a, b
            if valid_lat(a) and valid_lon(b):
                return b, a
            # fallback assume it's lon,lat
            return a, b

        # try two floats
        p2 = find_two_floats(s)
        if p2:
            a, b = p2
            # heuristics:
            if valid_lat(a) and valid_lon(b) and not (valid_lon(a) and valid_lat(b)):
                # a looks like lat, b looks like lon -> swap
                return b, a
            if valid_lon(a) and valid_lat(b) and not (valid_lat(a) and valid_lon(b)):
                # a lon, b lat
                return a, b
            # ambiguous
            if lon_first_flag:
                return a, b
            else:
                return b, a

    return None

def main():
    input_csv = "/content/reviews_classified_ALL_FEATURES.csv"
    output_geojson = "/content/reviews_classified_ALL_FEATURES.geojson"
    gps_col = "gps"
    lat_col = None
    lon_col = None
    lon_first_flag = False
    skip_invalid = True

    df = pd.read_csv(input_csv, dtype=str)  # read all as string to be safe

    # default gps column guess if not provided
    if not gps_col:
        for candidate in ['gps', 'location', 'coords', 'coordinate', 'geom', 'point']:
            if candidate in df.columns:
                gps_col = candidate
                break

    # default lat/lon candidates if user didn't specify
    if not lat_col or not lon_col:
        lat_candidates = ['lat', 'latitude', 'y']
        lon_candidates = ['lon', 'lng', 'long', 'longitude', 'x']
        if not lat_col:
            for c in lat_candidates:
                if c in df.columns:
                    lat_col = c
                    break
        if not lon_col:
            for c in lon_candidates:
                if c in df.columns:
                    lon_col = c
                    break

    features = []
    errors = 0

    for idx, row in df.iterrows():
        coords = detect_and_get_lon_lat(row, gps_col, lat_col, lon_col, lon_first_flag)
        if coords is None:
            errors += 1
            if skip_invalid:
                continue
            else:
                raise ValueError(
                    f"Could not parse coordinates for row {idx}. "
                    f"Try --gps-col/--lat-col/--lon-col or --lon-first. Row data: {row.to_dict()}"
                )

        lon, lat = coords

        if not (valid_lon(lon) and valid_lat(lat)):
            errors += 1
            if skip_invalid:
                continue
            else:
                raise ValueError(f"Coordinates out of range for row {idx}: lon={lon}, lat={lat}")

        # build properties
        props = {}
        for c in df.columns:
            if c == gps_col or c == lat_col or c == lon_col:
                continue

            val = row[c]
            if val is not None:
                val_str = str(val)
                if re.fullmatch(r'[-+]?\d+', val_str):
                    props[c] = int(val_str)
                else:
                    try:
                        f = float(val_str)
                        props[c] = f
                    except:
                        props[c] = val_str
            else:
                props[c] = None

        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [float(lon), float(lat)]
            },
            "properties": props
        }
        features.append(feature)

    fc = {"type": "FeatureCollection", "features": features}

    with open(output_geojson, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(features)} features to {output_geojson}. Rows skipped/failed: {errors}")

main()

Split CSV and import GEOJSON

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import re
import numpy as np

# Helper function to parse GPS string into lat, lng
def parse_gps_to_lat_lng(gps_str):
    if pd.isna(gps_str):
        return np.nan, np.nan

    s = str(gps_str).strip()

    # Try parsing "POINT(lon lat)" format (common for WKT)
    m = re.match(r'POINT\(\s*(-?\d+\.?\d*)\s+(-?\d+\.?\d*)\s*\)', s, re.IGNORECASE)
    if m:
        lon = float(m.group(1))
        lat = float(m.group(2))
        return lat, lon

    # If not WKT, try simple comma separated "lat,lng" or "lng,lat" with heuristics
    parts = s.split(',')
    if len(parts) == 2:
        try:
            val1 = float(parts[0].strip())
            val2 = float(parts[1].strip())

            # Heuristic: assume lat,lon if values are in expected ranges
            if -90 <= val1 <= 90 and -180 <= val2 <= 180:
                return val1, val2  # lat, lng

            # Or lon,lat
            if -180 <= val1 <= 180 and -90 <= val2 <= 90:
                return val2, val1  # lat, lng

        except ValueError:
            pass

    return np.nan, np.nan


# ---------------------------
# 1. read CSV
# ---------------------------
input_file = "/content/classified_output3000.csv"
df = pd.read_csv(input_file)

# ---------------------------
# 2. Process the coordinates and generate the 'lat' and 'lng' columns
# ---------------------------
df['lat'], df['lng'] = zip(*df['gps'].apply(parse_gps_to_lat_lng))

# Remove rows where lat or lng could not be parsed
df.dropna(subset=["lat", "lng"], inplace=True)

# ---------------------------
# 3. Create a Point geometry object
# ---------------------------
df["geometry"] = df.apply(lambda row: Point(row["lng"], row["lat"]), axis=1)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# ---------------------------
# 4. Divided into ‘visitors’ and ‘residents’
# ---------------------------
tourists = gdf[gdf["label"].str.lower() == "tourist"]
residents = gdf[gdf["label"].str.lower() == "resident"]

# ---------------------------
# 5. Save as CSV
# ---------------------------
tourists.to_csv("tourists.csv", index=False, encoding="utf-8-sig")
residents.to_csv("residents.csv", index=False, encoding="utf-8-sig")

# ---------------------------
# 6. Save as GeoJSON
# ---------------------------
tourists.to_file("tourists.geojson", driver="GeoJSON")
residents.to_file("residents.geojson", driver="GeoJSON")

print("Done!")